# FNPE minimal sanity check (controls-aware)

This notebook runs a tiny FNPE experiment using the updated control-aware
conditioning and normalization. It is intentionally small so you can
validate end-to-end wiring quickly.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, '..')

import numpy as np
import torch
import jax
import jax.numpy as jnp

from configs.config import ExperimentConfig
from methods.fnpe_method import FNPEMethod
from utils.normalization import Normalizer

print(f"JAX devices: {jax.devices()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


c:\Users\aritr\anaconda3\envs\sbi_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX devices: [CpuDevice(id=0)]
Using device: cuda


In [4]:
# Minimal config for a fast, end-to-end smoke test
cfg = ExperimentConfig(
    exp_name='fnpe_minimal_sanity',
    method='fnpe',
    T_seg=20,
    num_simulations=20,
    fnpe_window_size=2,
    fnpe_num_diffusion_steps=20,
    fnpe_score_fn_type='gauss_corrected',
    fnpe_steps_per_epoch=2,
    fnpe_max_obs_len=20,
    random_seed=42,
    no_plots=True,
)


method = FNPEMethod(
    cfg,
    prior=None,
    device=device,
    hidden_dim=32,
    num_hidden=2,
    model_type='gru',
    window_size=cfg.fnpe_window_size,
    num_epochs=1,
    steps_per_epoch=2,
    batch_size=16,
    num_diffusion_steps=20,
    score_fn_type='gauss_corrected',
    stop_after_epochs=1,
    validation_fraction=0.2,
    max_obs_len=cfg.fnpe_max_obs_len,
    proposal_type='naive',
    pilot_fraction=0.2,
    pilot_length=20,
    proposal_noise=0.03,
    gauss_posterior_precission_scale=2.0,
)

D_in = cfg.obs_dim + 4
method.build(input_dim=D_in, seq_len=cfg.T_seg)

print(f"Built FNPE method: {type(method).__name__}")


[FNPE] JAX devices: [CpuDevice(id=0)]
[FNPE] Default backend: cpu
Built FNPE method: FNPEMethod


In [5]:
# Train a tiny model
training_summary = method.train(
    num_simulations=cfg.num_simulations,
    T_obs=cfg.T_seg,
)

print('Training done:', training_summary)


[FNPE] Generating 20 training samples (T=2, window_size=2)...
[FNPE] (Full observation length T_obs=20 will be used at inference)
[FNPE] Using 'naive' proposal (covering full state space)
[FNPE] Proposal config: 10 pilots (=20.0% of 20), T_pilot=20, noise=0.03*std


Generating data: 100%|██████████| 1/1 [00:00<00:00,  1.13batch/s]


[FNPE] Data shapes: thetas=(20, 3), xs=(20, 2, 13)
[FNPE] Initializing SDE...
[FNPE] Training score network (max 1 epochs)...
[FNPE] Score network: 3,276 parameters
[FNPE] JIT compiling...
[FNPE] JIT compilation complete.
[FNPE] Starting training: 1 epochs, 2 steps/epoch


[FNPE] Epoch 1/1: Train=nan
[FNPE] Setting up sampler...
Training done: {'train_loss': [nan], 'final_train_loss': nan, 'epochs_trained': 1, 'train_time_s': 3.9232735633850098, 'num_simulations': 20, 'T_train': 2, 'T_obs_full': 20, 'budget_epochs': 1}


In [6]:
# Build normalizer from task stats (includes controls)
norm_stats = method.task.get_normalization_stats()
ctrl_mean = norm_stats.get('ctrl_mean')
ctrl_std = norm_stats.get('ctrl_std')
if ctrl_mean is None or ctrl_std is None:
    ctrl_mean = np.zeros(4, dtype=np.float32)
    ctrl_std = np.ones(4, dtype=np.float32)

normalizer = Normalizer(
    obs_mean=torch.tensor(np.array(norm_stats['obs_mean']), dtype=torch.float32),
    obs_std=torch.tensor(np.array(norm_stats['obs_std']), dtype=torch.float32),
    ctrl_mean=torch.tensor(np.array(ctrl_mean), dtype=torch.float32),
    ctrl_std=torch.tensor(np.array(ctrl_std), dtype=torch.float32),
    theta_mean=torch.tensor(np.array(norm_stats['theta_mean']), dtype=torch.float32),
    theta_std=torch.tensor(np.array(norm_stats['theta_std']), dtype=torch.float32),
).to(device)

posterior = method.build_posterior(normalizer=normalizer)
print('Posterior ready:', posterior is not None)


Posterior ready: True


In [7]:
# Smoke-test posterior sampling with controls included
key = jax.random.PRNGKey(cfg.random_seed + 123)
jax_prior = method.task.get_prior()
simulator_fn = method.task.get_simulator()

key, key_theta, key_sim = jax.random.split(key, 3)
theta_true = jax_prior.sample(key_theta, (1,))[0]
x_phys = simulator_fn(key_sim, theta_true, cfg.T_seg)

x_phys_torch = torch.tensor(np.array(x_phys), dtype=torch.float32).unsqueeze(0).to(device)
x_norm = normalizer.normalize_x(x_phys_torch, cfg.obs_dim)

theta_post_norm = posterior.sample((10,), x=x_norm)
theta_post_phys = normalizer.unnormalize_theta(theta_post_norm)

print('x_phys shape:', x_phys.shape)
print('x_norm shape:', x_norm.shape)
print('posterior samples:', theta_post_phys.shape)


x_phys shape: (20, 13)
x_norm shape: torch.Size([1, 20, 13])
posterior samples: torch.Size([10, 3])


In [8]:
# Quick PPC plot check (synthetic PPC around the simulated trajectory)
from utils.plots import plot_ppc_trajectories

rng = np.random.default_rng(0)
y_real = np.array(x_phys[:, : cfg.obs_dim], dtype=np.float32)
K = 50
noise = rng.normal(scale=0.05, size=(K,) + y_real.shape).astype(np.float32)
y_ppc = y_real[None, :, :] + noise

fig_dir = Path('../experiments/fnpe_minimal_sanity/figures')
fig_dir.mkdir(parents=True, exist_ok=True)
plot_ppc_trajectories(
    y_real=y_real,
    y_ppc=y_ppc,
    obs_labels=[f'obs_{i}' for i in range(cfg.obs_dim)],
    dt=cfg.dt,
    out_path=fig_dir / 'ppc_test.png',
    max_trajs=30,
)
print(f"PPC plot saved to: {fig_dir / 'ppc_test.png'}")


[plots] Saved PPC trajectories to ..\experiments\fnpe_minimal_sanity\figures\ppc_test.png
PPC plot saved to: ..\experiments\fnpe_minimal_sanity\figures\ppc_test.png
